<a href="https://colab.research.google.com/github/Nabhit03/GIS-based-Waste-Collection-Route-Optimization-using-AI/blob/main/Run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q osmnx networkx scikit-learn ortools pandas numpy shapely folium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 12.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.


In [76]:
from google.colab import files
uploaded = files.upload()  # a file picker will pop up — select waste-route-optimizer.zip

Saving waste-route-optimizer_updated copy 2.zip to waste-route-optimizer_updated copy 2.zip


In [77]:
import zipfile, os, shutil

zip_name = list(uploaded.keys())[0]

# Remove any old extracted folder so nothing stale survives
if os.path.exists('waste-route-optimizer'):
    shutil.rmtree('waste-route-optimizer')

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')

%cd waste-route-optimizer
!ls

/content/waste-route-optimizer/waste-route-optimizer/waste-route-optimizer/waste-route-optimizer/waste-route-optimizer/waste-route-optimizer
outputs  README.md  requirements.txt  src  {src,data,outputs}


In [78]:
print("Connectivity fix present:", "strongly_connected" in open("src/network_utils.py").read())
print("Tile fix present:", "OpenStreetMap" in open("src/visualize.py").read())

Connectivity fix present: True
Tile fix present: True


In [79]:
import re

with open('src/config.py', 'r') as f:
    cfg = f.read()

cfg = re.sub(r'NUM_COLLECTION_POINTS = \d+', 'NUM_COLLECTION_POINTS = 250', cfg)
cfg = re.sub(r'NUM_TRUCKS = \d+', 'NUM_TRUCKS = 6', cfg)
cfg = re.sub(r'TRUCK_CAPACITY = \d+', 'TRUCK_CAPACITY = 80', cfg)
cfg = re.sub(r'NUM_COLLECTION_POINTS = \d+', 'NUM_COLLECTION_POINTS = 100', cfg)

with open('src/config.py', 'w') as f:
    f.write(cfg)

print("Config updated. Current settings:")
!grep -E "CITY_NAME|NUM_TRUCKS|TRUCK_CAPACITY|NUM_COLLECTION_POINTS" src/config.py

Config updated. Current settings:
CITY_NAME = "Chandigarh, India"   # used if OSMnx/internet is available
NUM_TRUCKS = 6
TRUCK_CAPACITY = 80          # max demand units a truck can service per trip
# Minimum stops per vehicle = this fraction * (total stops / NUM_TRUCKS), rounded
NUM_COLLECTION_POINTS = 100


In [80]:
!python -m src.main

[network_utils] Road network was not fully connected. Kept largest strongly-connected component: 11960/12148 nodes retained.
[network_utils] Loaded real OSM road network for 'Chandigarh, India' (11960 nodes, 30731 edges).
[data_prep] Generated 85 unique collection stops (33 commercial, 52 residential).
[clustering] K-Means initial zone sizes (stops per truck): {0: 9, 1: 16, 2: 13, 3: 15, 4: 15, 5: 17}
[visualize] Saved zone map -> outputs/zones_map.html
[visualize] Saved route map -> outputs/tier1_naive_routes.html
[visualize] Saved route map -> outputs/tier2_clustered_tsp_routes.html
[vrp_solver] Auto-scaled workload balance weight -> 78 (avg stop-to-stop time=13.1 min x multiplier=6)
[vrp_solver] WARNING: could not find a feasible solution with a minimum of 7 stops/vehicle enforced (ENFORCE_MIN_STOPS_PER_VEHICLE in config.py). This can happen when capacity or time-window constraints don't allow an even split. Falling back to balancing-only (no hard minimum) -- some trucks may still e

In [81]:
import pandas as pd
pd.read_csv("outputs/comparison_metrics.csv")

,strategy,total_time_min,approx_distance_km,overlap_edges,workload_std_stops,fuel_liters,co2_kg,tw_violations,% reduction vs Tier 1 (naive)
0,Tier 1: Naive NN per zone,644.0,268.33,0,2.61,76.67,205.47,0,0.0
1,Tier 2: K-Means + TSP per zone,613.0,255.42,0,2.61,72.98,195.58,0,4.8
2,Tier 3: Joint CVRP + Time Windows (final),594.0,247.50,0,10.07,70.71,189.51,0,7.8


In [82]:
import html
from IPython.display import HTML, display

def show_map(path, width=900, height=600):
    with open(path, 'r') as f:
        content = f.read()
    escaped = html.escape(content)
    iframe = f'<iframe srcdoc="{escaped}" width="{width}" height="{height}" style="border:none;"></iframe>'
    display(HTML(iframe))

show_map("outputs/zones_map.html")

/usr/local/lib/python3.13/dist-packages/IPython/core/display.py:724: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


In [83]:
show_map("outputs/tier1_naive_routes.html")

In [84]:
show_map("outputs/tier2_clustered_tsp_routes.html")

In [85]:
show_map("outputs/tier3_optimized_routes.html")

In [86]:
!zip -rq outputs.zip outputs
files.download("outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>